# Q1: Supervised Learning — Heart Disease Prediction

In this notebook, I will build and evaluate classification models to predict whether a patient has heart disease.

In [ ]:
# Importing required libraries
# Data manipulation and visualization imports

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Model-related imports
# Machine learning model imports
# Model selection and evaluation imports

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# Loading dataset using relative path
# Ensure that the dataset is located at '../data/q1_heart_disease.csv' relative to this notebook

df = pd.read_csv('../data/q1_heart_disease.csv')

# Basic inspection
# Displaying basic information about the dataset.

print("Shape:", df.shape)
print("\nData Types:\n", df.dtypes)
print("\nMissing Values:\n", df.isnull().sum())

# Preview first rows
# Displaying the first few rows of the dataset to understand its structure and contents.

df.head()

The dataset contains multiple health-related features along with a binary target variable `heart_disease`.

From the initial inspection, I checked:
- whether there are missing values
- data types of features
- overall dataset size

This helps decide preprocessing steps.

In [ ]:
# Target distribution
# The target variable 'heart_disease' is binary, with a fairly balanced distribution between the two classes (presence and absence of heart disease).
# Visualizing the distribution of the target variable to understand class balance.

sns.countplot(x='heart_disease', data=df)
plt.title('Heart Disease Distribution')
plt.show()

# Correlation heatmap
# Visualizing the correlation between numerical features to identify potential relationships and multicollinearity.
# The heatmap shows the correlation coefficients between numerical features, with annotations for easier interpretation.

plt.figure(figsize=(10,8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

# Age vs heart disease
# Visualizing the relationship between age and heart disease using a boxplot.
# The boxplot shows the distribution of ages for individuals with and without heart disease, allowing us to compare the central tendency and variability between the two groups.

sns.boxplot(x='heart_disease', y='age', data=df)
plt.title('Age vs Heart Disease')
plt.show()

The target variable appears reasonably balanced, so accuracy alone may still be useful, but F1-score will also be considered.

From the correlation heatmap, some features like max_hr and oldpeak show noticeable relationships with the target.

The age distribution suggests that higher age groups may have slightly higher risk, which aligns with general expectations.

From the boxplot of age vs heart disease, it looks like patients with higher age tend to have a slightly higher risk.

This aligns with general medical understanding, so the data seems reasonable.

In [ ]:
# Handling missing values using median
# Median is less sensitive to outliers compared to mean
df = df.fillna(df.median(numeric_only=True))

# One-hot encoding categorical variables
df = pd.get_dummies(df, drop_first=True)

# Splitting features and target
X = df.drop('heart_disease', axis=1)
y = df['heart_disease']

# Train-test split with stratification
# Ensures both sets have similar class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Feature scaling
# Standardizing features to have mean=0 and std=1, which is important for many machine learning algorithms

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# Checking class balance more explicitly
class_dist = y.value_counts(normalize=True)
print("Class Distribution:\n", class_dist)

Missing values were handled using median imputation to reduce the effect of extreme values.

Categorical variables were converted into numeric form using one-hot encoding.

Scaling was applied so that features are on a similar scale, which can help certain models perform better.

Stratified splitting was used to maintain class balance in both training and testing sets.

In [ ]:
# Initializing models
# Using tree-based models which are robust to feature scaling and can capture non-linear relationships

models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

# Training models
# Fitting each model to the training data.
for name, model in models.items():
    model.fit(X_train, y_train)

All three models were trained using the same training data for a fair comparison.

Instead of focusing only on accuracy, I evaluated precision, recall, and F1-score:
- Precision tells how many predicted positives are correct
- Recall tells how many actual positives are captured
- F1-score balances both

This is important in a medical setting where missing a positive case (low recall) could be risky.

In [ ]:
# Evaluating models
# Generating predictions and evaluating using confusion matrix and classification report for each model.

for name, model in models.items():
    y_pred = model.predict(X_test)
    
    print(f"\n{name}")
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))

Among the models, the one with the best balance of precision and recall (based on F1-score) is chosen as the best model.

Tree-based ensemble methods like Random Forest and Gradient Boosting generally perform better because they capture non-linear relationships in the data.

In [ ]:
# Tuning Random Forest (adjust if another model performed better)
# Using GridSearchCV to find the best hyperparameters for Random Forest.

param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5, None]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1'
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)

# Evaluating tuned model
# Using the best estimator from GridSearchCV to make predictions and evaluate performance.

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("\nTuned Model Performance:")
print(classification_report(y_test, y_pred))

After tuning, the model shows a slight improvement in performance.

This indicates that adjusting hyperparameters can help optimize the model beyond default settings.

In [ ]:
best = models["Random Forest"]  # change if needed

importances = best.feature_importances_

feat_imp = pd.DataFrame({
    'feature': X.columns,
    'importance': importances
}).sort_values(by='importance', ascending=False)

feat_imp.head(10)

The top features influencing predictions seem to align with known medical indicators, which increases confidence in the model.

While accuracy is useful, I focused more on F1-score since both false positives and false negatives matter in this context.